# Purpose/risk prompt probe

Try the current `purpose_risk` prompt on a small set of labels and read what it
extracts. Pick a label set (any set notebook 06 or a draw script saved) and,
optionally, a subset of it: explicit `IDS`, or the first `LIMIT` after `ORDER`.

`PurposeRiskService` has the store create a run over the selection and streams every
label through the model. One row per label, failed ones included, goes to
`data/dsld/runs/<label set>-<YYMMDDHHMM>-<n>/raw/purpose_risk.jsonl`. The prompt text is
part of the run's `config_hash`, so each prompt edit is its own run, and the last
section compares the newest run with the one before it.

The running log is the controller's and the model's `logging` output, sent to this
notebook: one progress line per finished label with the ETA, a line per retry, and a
warning per label given up on.

In [1]:
import logging
import re
import sys

import pandas as pd

from engine.repositories.dsld.labels import loader as label_sets
from engine.repositories.dsld.runs import loader as runs
from engine.repositories.dsld.runs import paths as run_paths
from engine.services.purpose_risk.service import PurposeRiskService

LABEL_SET = "multiverb50-seed20261006-r3"
IDS = None      # e.g. [297325, 44299] -- every id must be in LABEL_SET
LIMIT = None    # e.g. 10 -- the first LIMIT after ORDER
ORDER = None    # None (file order), "sorted", or "random" (needs SEED)
SEED = None

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    stream=sys.stdout,
    force=True,
)

## The label set

What the run will read: the cleaned, flattened labels, the set's manifest, and the
`passages_sha` the run records.

In [2]:
import json

from engine.repositories.dsld.labels import paths as label_paths

passages_sha, labels = label_sets.load_flattened(LABEL_SET)
print(passages_sha, len(labels))
print(json.loads((label_paths.set_dir(LABEL_SET) / label_paths.MANIFEST_NAME).read_text()).get("description", ""))

c5950d2d7c42e448 50
probe set for the one-verb-per-claim rule: 25 labels with a sentence holding >=2 distinct purpose verbs, 25 with >=2 risk verbs (and none of the purpose kind). 'help(s) (to)' and 'may' fold into the next verb; at most one label per distinct sentence, and at most 4 of the stock 'discontinue use and consult ... if adverse reactions occur' kind. Drawn from 6000 dump ids shuffled with seed 20261006, excluding every id in an existing label set. Not annotated.


## Is the GPU free?

The model needs ~21 GB. Anything listed under compute apps is holding the card.

In [3]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

memory.free [MiB]
24111 MiB
pid, process_name, used_gpu_memory [MiB]


## The run

The selection above, over the set. Leaving the `with` block closes the controller,
which stops the server and frees the card -- also on an error or an interrupt. The
last line printed is the run directory and its counts.

In [7]:
with PurposeRiskService() as svc:
    svc.run(LABEL_SET, ids=IDS, limit=LIMIT, order=ORDER, seed=SEED)

2026-10-06 17:19:05,143 INFO engine.controllers.qwen_controller 1/50 633 tok · 6.2 tok/s · eta 1m35s
2026-10-06 17:19:07,124 INFO engine.controllers.qwen_controller 2/50 1676 tok · 24.8 tok/s · eta 1m34s
2026-10-06 17:19:07,771 INFO engine.controllers.qwen_controller 3/50 2670 tok · 42.3 tok/s · eta 1m11s
2026-10-06 17:19:09,815 INFO engine.controllers.qwen_controller 4/50 3797 tok · 40.3 tok/s · eta 1m16s
2026-10-06 17:19:11,468 INFO engine.controllers.qwen_controller 5/50 4687 tok · 38.6 tok/s · eta 1m14s
2026-10-06 17:19:16,012 INFO engine.controllers.qwen_controller 6/50 6171 tok · 49.9 tok/s · eta 1m34s
2026-10-06 17:19:17,192 INFO engine.controllers.qwen_controller 7/50 7674 tok · 71.8 tok/s · eta 1m26s
2026-10-06 17:19:18,783 INFO engine.controllers.qwen_controller 8/50 9606 tok · 91.9 tok/s · eta 1m22s
2026-10-06 17:19:18,789 INFO engine.controllers.qwen_controller 9/50 10516 tok · 95.9 tok/s · eta 1m11s
2026-10-06 17:19:21,011 INFO engine.controllers.qwen_controller 10/50 1132

## The rows

The newest run over the set, one row per label.

In [5]:
def runs_of(label_set):
    """Every run over `label_set`, oldest first."""
    return sorted(
        run_paths.RUNS_ROOT.glob(f"{label_set}-*"),
        key=lambda d: (d.name.rsplit("-", 2)[1], int(d.name.rsplit("-", 1)[1])),
    )


run = runs_of(LABEL_SET)[-1]
print(run)
print(runs.read_manifest(run))

df = pd.DataFrame(runs.read_rows(run))
print(df.status.value_counts().to_string())
df[["dsld_id", "status", "attempts", "failed", "purpose", "risk", "latency_ms"]]

/var/tmp/ge74wan/repos/supplement-kg/data/dsld/runs/multiverb50-seed20261006-r3-2610061457-0
{'prompt': 'purpose_risk', 'label_set': 'multiverb50-seed20261006-r3', 'passages_sha': 'c5950d2d7c42e448', 'selection': {'ids': None, 'limit': None, 'order': None, 'seed': None}, 'config_hash': '997cfa8a60219618'}
status
ok    50


,dsld_id,status,attempts,failed,purpose,risk,latency_ms
0,19629,ok,1,[],[{'text': 'Supports Nutritional Balance and Me...,[],2014
1,20038,ok,1,[],"[{'text': 'supports renewal of cartilage'}, {'...",[{'text': 'Pregnant or nursing women and indiv...,4537
2,25984,ok,1,[],[],[{'text': 'Do not exceed four (4) softgels in ...,3365
3,30158,ok,1,[],"[{'text': 'Supports Immune and Bone Health'}, ...","[{'text': 'If you are pregnant, nursing, takin...",3221
4,35775,ok,1,[],[{'text': 'Essential for healthy heart functio...,"[{'text': 'If you are pregnant or nursing, tak...",1594
5,16518,ok,1,[],[{'text': 'Users of Tokkyo Test Extreme report...,"[{'text': 'DO NOT EXCEED RECOMMENDED DOSAGE'},...",13568
6,39924,ok,1,[],[],"[{'text': 'Do not use if you are pregnant, may...",1838
7,19865,ok,1,[],"[{'text': 'Prescription Potency Fat Burner'}, ...",[{'text': 'Consult a physician or licensed qua...,15831
8,31215,ok,1,[],"[{'text': 'Incomparable Training Intensity'}, ...",[{'text': 'NAPALM contains caffeine and other ...,11973
9,40508,ok,1,[],"[{'text': 'sharpen learning skills'}, {'text':...",[],2575


## Run stats

Status and attempts, items per label, how many items are copied exactly from the
label, and tokens and latency per label.

In [6]:
# Status and attempts: how many labels needed a retry, and how many never passed.
print(df.status.value_counts().to_string(), "\n")
print(df.attempts.value_counts().sort_index().rename("labels by attempts").to_string(), "\n")

# How much each label yielded: items per label, and labels with none.
df["n_purpose"] = df.purpose.str.len()
df["n_risk"] = df.risk.str.len()
print(df[["n_purpose", "n_risk"]].describe().round(2).to_string(), "\n")
print(f"no purpose: {(df.n_purpose == 0).sum()}   no risk: {(df.n_risk == 0).sum()}   "
      f"neither: {((df.n_purpose == 0) & (df.n_risk == 0)).sum()}   of {len(df)}\n")

# Copied exactly: each item must be a substring of its passage (whitespace collapsed).
squash = lambda s: re.sub(r"\s+", " ", s).strip()
not_copied = [
    (row.dsld_id, kind, item["text"])
    for row in df.itertuples() for kind in ("purpose", "risk")
    for item in getattr(row, kind)
    if squash(item["text"]) not in squash(labels[row.dsld_id].passage)
]
n_items = int(df.n_purpose.sum() + df.n_risk.sum())
print(f"copied exactly: {n_items - len(not_copied)} / {n_items}")
for x in not_copied:
    print("  not in label:", x)
print()

# Tokens and latency per label. Slots run in parallel, so summed latency is not wall time.
print(df[["prompt_tokens", "completion_tokens", "latency_ms"]].describe().round(0).to_string())

status
ok    50 

attempts
1    50 

       n_purpose  n_risk
count      50.00   50.00
mean        3.62    3.48
std         3.14    2.90
min         0.00    0.00
25%         1.25    2.00
50%         2.50    2.00
75%         5.00    4.75
max        15.00   12.00 

no purpose: 2   no risk: 2   neither: 0   of 50

copied exactly: 353 / 355
  not in label: (58314, 'purpose', 'POWERED BY elevATP & TORABOLIC HEALTHY MITOCHONDRIAL ENERGY')
  not in label: (67383, 'purpose', 'support your immune system with ViruTech')

       prompt_tokens  completion_tokens  latency_ms
count           50.0               50.0        50.0
mean           872.0              132.0      5223.0
std            279.0              107.0      3380.0
min            489.0               12.0      1594.0
25%            696.0               64.0      3036.0
50%            790.0               92.0      3754.0
75%            969.0              158.0      6682.0
max           2012.0              466.0     15831.0


## Purpose and risk per label

Each label's passage beside the purpose and risk items the model took from it. Set
`SHOW` to a list of ids to read those, or leave it None for a seeded sample.

In [7]:
SHOW = None     # e.g. [297325, 44299]
SAMPLE_N, SAMPLE_SEED = 5, 0

shown = df[df.dsld_id.isin(SHOW)] if SHOW else df.sample(min(SAMPLE_N, len(df)), random_state=SAMPLE_SEED)
for row in shown.itertuples():
    label = labels[row.dsld_id]
    print(f"=== {row.dsld_id}  {label.name}  [{row.status}, {row.attempts} attempt(s)]")
    print(label.passage[:800] + (" ..." if len(label.passage) > 800 else ""))
    print("--- purpose")
    for item in row.purpose:
        print(f"  {item['text']}")
    print("--- risk")
    for item in row.risk:
        print(f"  {item['text']}")
    print()

NameError: name 'labels' is not defined

## Most repeated purpose and risk items

In [ ]:
# The most repeated items across the run. Repeated risk items are mostly generic
# label warnings; a later filter handles them on the output side.
for kind in ("purpose", "risk"):
    counts = pd.Series([item["text"] for items in df[kind] for item in items]).value_counts()
    print(f"--- {kind}: {counts.sum()} items, {len(counts)} distinct")
    print(counts.head(10).to_string(), "\n")

## One failed label in full

The raw answer is kept on every row, so a failure can be read without the GPU.

In [ ]:
failed = df[df.status != "ok"]
failed.iloc[0].to_dict() if len(failed) else "no failed labels"

## Compared with the previous run

What changed between the run before and the newest one over the same set -- usually a
prompt edit. Only labels in both runs are compared; items match ignoring case and
punctuation.

In [ ]:
previous = runs_of(LABEL_SET)[-2] if len(runs_of(LABEL_SET)) > 1 else None
if previous is None:
    print("only one run over this set")
else:
    print("previous:", previous.name, "\nnewest:  ", run.name, "\n")
    old = {r["dsld_id"]: r for r in runs.read_rows(previous)}
    key = lambda s: " ".join(re.findall(r"\w+", s.lower()))
    n_changed = 0
    for row in df.itertuples():
        if row.dsld_id not in old:
            continue
        for kind in ("purpose", "risk"):
            before = [item["text"] for item in old[row.dsld_id][kind]]
            after = [item["text"] for item in getattr(row, kind)]
            gone = [t for t in before if key(t) not in {key(a) for a in after}]
            new = [t for t in after if key(t) not in {key(b) for b in before}]
            if gone or new:
                n_changed += 1
                print(f"{row.dsld_id} {kind}")
                for t in gone:
                    print(f"  - {t}")
                for t in new:
                    print(f"  + {t}")
    print(f"\n{n_changed} fields changed")

## Resume

If the run was interrupted, this reads the run's ids and label set from the run
directory and does only the ids without a row yet. A changed prompt or sampling
setting is refused -- start a new run instead.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(run=run)